**OVERFITTING AND CROSS VALIDATION**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("train.csv")

numeric = ["GrLivArea", "OverallQual", "GarageCars", "TotalBsmtSF", "YearBuilt", "FullBath", "BedroomAbvGr"]
X = pd.get_dummies(df[numeric + ["Neighborhood"]], dtype=int)
y = df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rows = []
for d in [1, 2, 3, 5, 8, 12, 20, None]:
  tree = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_train, y_train)
  rows.append({
      "max_depth": "no limit" if d is None else d,
      "train MAE": round(mean_absolute_error(y_train, tree.predict(X_train))),
      "test MAE": round(mean_absolute_error(y_test, tree.predict(X_test))),
  })

pd.DataFrame(rows)

,max_depth,train MAE,test MAE
0,1,42613,46156
1,2,34307,34569
2,3,29661,30450
3,5,22161,25355
4,8,12922,24314
5,12,4360,25127
6,20,451,24502
7,no limit,113,25110


*As the tree gets deeper, the training error falls from 42,613 to 113, but the test error stops improving around depth 8 (24,314) and then stays near 25,000. With no depth limit, the tree is off by about 113 on houses it has seen and about 25,110 on new ones, so it has memorized the training houses. A large gap between training and test error is the sign of overfitting.*

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor

cv = KFold(n_splits=5, shuffle=True, random_state=42)

candidates = {
    "Baseline (guess the average)": DummyRegressor(strategy="mean"),
    "Linear regression": LinearRegression(),
    "Decision tree, depth 5": DecisionTreeRegressor(max_depth=5, random_state=42),
    "Decision tree, no limit": DecisionTreeRegressor(random_state=42),
    "Gradient boosting": GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)
}

out = []
for name, m in candidates.items():
  scores = -cross_val_score(m, X, y, cv=cv, scoring= "neg_mean_absolute_error")
  out.append({"Model": name,
              "Average MAE": round(scores.mean()),
              "Lowest fold": round(scores.min()),
              "Highest fold": round(scores.max())})

pd.DataFrame(out)

,Model,Average MAE,Lowest fold,Highest fold
0,Baseline (guess the average),57458,53430,62576
1,Linear regression,22333,19921,25231
2,"Decision tree, depth 5",26512,24351,30306
3,"Decision tree, no limit",26830,25110,29640
4,Gradient boosting,18990,17547,22097


*With 5-fold cross-validation, gradient boosting has an average error of 18,990 (folds from 17,547 to 22,097), about 15% lower than linear regression (22,333) and about 67% lower than guessing the average (57,458). The unlimited decision tree is about 41% worse than gradient boosting. The single-split score for gradient boosting (17,635) was slightly optimistic, and the fold-to-fold spread (about 4,500) means differences of 1,000 to 2,000 between models are within noise.*